# M01 lab solution: data warm-up

**Brief.** Load a CSV into pandas and into SQLite, answer five questions both ways, and expose one of them through a small Flask endpoint.

The data is `../data/churn_prediction.csv` (15,929 bank customers, 21 columns), the file notebook `03b` of this module explores. Nothing here calls a model, so no API key is needed.

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

sys.path.insert(0, str(Path("../../../../src").resolve()))
load_dotenv()

DATA = Path("../data")

import sqlite3

import numpy as np
import pandas as pd
from flask import Flask, jsonify, request

## 1. Load the CSV into pandas and into SQLite

`to_sql` writes `NaN` as `NULL`, so the missing `gender`, `dependents`, `occupation` and `city` values stay missing in both copies. Check the shape and the missing counts before trusting any answer.

In [ ]:
df = pd.read_csv(DATA / "churn_prediction.csv")
df.info()

con = sqlite3.connect(":memory:")
df.to_sql("customers", con, index=False)
print(con.execute("SELECT COUNT(*) FROM customers").fetchone()[0], "rows in SQLite")

## 2. Five questions, answered both ways

Each question has a pandas answer and a SQL answer, and an assertion that they agree. Values that reach SQL from outside go in as parameters (`?`), never through an f-string.

In [ ]:
def same(a: pd.DataFrame, b: pd.DataFrame) -> None:
    """Assert two answers agree: same columns, same rows in the same order, floats equal to 1e-9."""
    pd.testing.assert_frame_equal(a.reset_index(drop=True), b.reset_index(drop=True),
                                  check_dtype=False, check_exact=False, rtol=1e-9)

answers = {}

In [ ]:
# Q1. How many customers are there, and how many churned?
q1_pd = pd.DataFrame({"customers": [len(df)], "churned": [int(df["churn"].sum())]})
q1_sql = pd.read_sql("SELECT COUNT(*) AS customers, SUM(churn) AS churned FROM customers", con)
same(q1_pd, q1_sql)
answers["Q1 customers and churned"] = q1_pd
q1_pd

In [ ]:
# Q2. Churn rate by gender (rows with no gender left out, as groupby does by default).
q2_pd = (df.groupby("gender", as_index=False)["churn"].mean()
           .rename(columns={"churn": "churn_rate"}).sort_values("gender"))
q2_sql = pd.read_sql("""
    SELECT gender, AVG(churn) AS churn_rate
    FROM customers WHERE gender IS NOT NULL
    GROUP BY gender ORDER BY gender""", con)
same(q2_pd, q2_sql)
answers["Q2 churn rate by gender"] = q2_pd
q2_pd

In [ ]:
# Q3. Average current balance and customer count per occupation.
q3_pd = (df.groupby("occupation", as_index=False)
           .agg(customers=("customer_id", "size"), avg_balance=("current_balance", "mean"))
           .sort_values("occupation"))
q3_sql = pd.read_sql("""
    SELECT occupation, COUNT(*) AS customers, AVG(current_balance) AS avg_balance
    FROM customers WHERE occupation IS NOT NULL
    GROUP BY occupation ORDER BY occupation""", con)
same(q3_pd, q3_sql)
answers["Q3 balance by occupation"] = q3_pd
q3_pd

In [ ]:
# Q4. Churn rate of customers at or above an age, passed as a parameter.
def churn_rate_pd(min_age: int) -> float:
    return float(df.loc[df["age"] >= min_age, "churn"].mean())

def churn_rate_sql(min_age: int) -> float:
    return con.execute("SELECT AVG(churn) FROM customers WHERE age >= ?", (min_age,)).fetchone()[0]

MIN_AGE = 60
q4_pd = pd.DataFrame({"min_age": [MIN_AGE], "churn_rate": [churn_rate_pd(MIN_AGE)]})
q4_sql = pd.DataFrame({"min_age": [MIN_AGE], "churn_rate": [churn_rate_sql(MIN_AGE)]})
same(q4_pd, q4_sql)
answers["Q4 churn rate at age >= 60"] = q4_pd
q4_pd

In [ ]:
# Q5. The five cities with the most customers (ties broken by city code, so both orders agree).
q5_pd = (df.dropna(subset=["city"]).groupby("city", as_index=False).size()
           .rename(columns={"size": "customers"})
           .sort_values(["customers", "city"], ascending=[False, True]).head(5))
q5_sql = pd.read_sql("""
    SELECT city, COUNT(*) AS customers
    FROM customers WHERE city IS NOT NULL
    GROUP BY city ORDER BY customers DESC, city ASC LIMIT 5""", con)
same(q5_pd, q5_sql)
answers["Q5 top cities"] = q5_pd
q5_pd

## 3. Serve Q4 through a Flask route

`app.test_client()` sends requests to the app in this process, so no server is left running. The route validates its input and returns 400 on a bad value.

In [ ]:
app = Flask(__name__)

@app.get("/churn-rate")
def churn_rate_route():
    try:
        min_age = int(request.args.get("min_age", "0"))
    except ValueError:
        return jsonify(error="min_age must be an integer"), 400
    return jsonify(min_age=min_age, churn_rate=churn_rate_sql(min_age))

with app.test_client() as client:
    ok = client.get(f"/churn-rate?min_age={MIN_AGE}")
    bad = client.get("/churn-rate?min_age=sixty")

print(ok.status_code, ok.get_json())
print(bad.status_code, bad.get_json())
assert ok.status_code == 200 and np.isclose(ok.get_json()["churn_rate"], churn_rate_pd(MIN_AGE))
assert bad.status_code == 400

## Results

In [ ]:
results = pd.DataFrame([
    {"question": q, "rows": len(a), "pandas == SQL": True} for q, a in answers.items()
] + [{"question": "Flask GET /churn-rate?min_age=60", "rows": 1, "pandas == SQL": True}])
results

## What this shows

- pandas and SQL give the same answers only when they treat missing values the same way: `groupby` drops `NaN` keys, so the SQL needs `WHERE ... IS NOT NULL`.
- A row order is part of an answer. Both sides sort on every column that can tie.
- The Flask route reuses the parameterised query, and `test_client()` checks it without starting a server.